# Day 50: The AI Study Buddy 🎓 (Capstone)

**Day 50 of 50.** Everything we built becomes *one* LangGraph agent — a study buddy that teaches you **LangChain itself**, from the real docs.

The whole series, assembled:
- 🧠 **Memory** (Day 49, LangMem) — it learns which LangChain topics you're weak at
- 🔎 **RAG** (Days 8-14) — it recalls the right memory by meaning
- 🔌 **MCP tools** (Days 35-36) — it reads the **live LangChain docs** over MCP
- 🛠️ **Tool calling** — a LangGraph agent decides when to look things up
- 📜 **Skills** — it loads a "tutor" skill that shapes how it teaches
- ⚡ **Prompt caching** (Days 39-40) — it reuses the fetched doc across quiz, explain, and grade
- ✅ **Judge** (Day 45) — it grades you honestly
- 🎙️ **Voice** (Day 48) — it talks

The series taught you LangChain. The finale is a buddy that keeps teaching it — so you don't need me.

## Install

In [ ]:
%pip install langmem langchain-google-genai langchain-mcp-adapters langgraph google-genai numpy --quiet

## Setup

In [15]:
import os, json, warnings, logging
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)   # silence the genai MCP-schema 'additionalProperties' notices
from pathlib import Path
from datetime import date
from typing import Literal
import numpy as np
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langmem import create_memory_manager
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash")  # Gemini 3.5 Flash
embed = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
print("✅ model + embedder ready")

✅ model + embedder ready


## Piece 1 — Memory: it learns you (LangMem, Day 49)

I don't type up a profile — I just talk to it. **LangMem** reads one short chat and extracts atomic, typed memories: which LangChain topics I'm weak at, how I like answers, what I've built.

In [2]:
class MemoryItem(BaseModel):
    content: str = Field(description="one atomic memory about the student")
    type: Literal["semantic", "episodic", "procedural"] = Field(
        description="semantic = a durable fact about the student; "
                    "episodic = a specific thing that happened; "
                    "procedural = a preference for how the buddy should respond")

manager = create_memory_manager(
    "google_genai:gemini-3.5-flash",
    schemas=[MemoryItem],
    instructions=("Extract ALL distinct memories about the student, one atomic item each. "
                  "Capture their name/background (semantic), EVERY LangChain topic they say they "
                  "struggle with (semantic), ANY preference about how answers should be given "
                  "(procedural), and ANY specific thing they built or did (episodic). Do not merge items."))

onboarding = [
    {"role": "user", "content": "Hi, I'm Yash - I've been learning LangChain for 50 days. I'm still shaky on LangGraph state and on tool calling, and I like short, example-first explanations. Yesterday I shipped my first RAG bot."},
    {"role": "assistant", "content": "Love it, Yash - short and example-first, got it."},
]

learned = manager.invoke({"messages": onboarding})
for m in learned:
    print(f"[{m.content.type:<10}] {m.content.content}")

[semantic  ] Name is Yash
[semantic  ] Has been learning LangChain for 50 days
[semantic  ] Struggles with LangGraph state
[semantic  ] Struggles with tool calling
[procedural] Prefers short, example-first explanations
[episodic  ] Shipped their first RAG bot yesterday


## Piece 2 — RAG: searchable memory (Days 8-14)

We embed each memory so the buddy can recall the *relevant* ones by meaning — the Day 8-14 retrieval idea, pointed at memory — plus a small de-dupe so it never stores the same fact twice.

In [3]:
MEMORY_FILE = Path("day50_memory.md")

class Memory:
    """Typed memories the buddy can search by meaning (embeddings)."""
    def __init__(self, embedder):
        self.embed = embedder
        self.items = []  # each: {type, content, vec}

    def add_many(self, mems):
        vecs = self.embed.embed_documents([m.content.content for m in mems])
        for m, v in zip(mems, vecs):
            v = np.array(v)
            if self.items:  # skip near-duplicates
                sims = [float(v @ it["vec"] / (np.linalg.norm(v) * np.linalg.norm(it["vec"]))) for it in self.items]
                if max(sims) > 0.88:
                    continue
            self.items.append({"type": m.content.type, "content": m.content.content, "vec": v})

    def recall(self, query, k=1):
        q = np.array(self.embed.embed_query(query))
        sim = lambda v: float(v @ q / (np.linalg.norm(v) * np.linalg.norm(q)))
        return sorted(self.items, key=lambda it: sim(it["vec"]), reverse=True)[:k]

    def preferences(self):
        return [it["content"] for it in self.items if it["type"] == "procedural"]

    def retire(self, predicate):
        self.items = [it for it in self.items if not predicate(it)]

    def to_markdown(self):
        sections = [("semantic", "Facts about you (semantic)"),
                    ("episodic", "Things that happened (episodic)"),
                    ("procedural", "How to help you (procedural)")]
        lines = ["# 🧠 What the buddy remembers about you\n"]
        for key, title in sections:
            items = [it["content"] for it in self.items if it["type"] == key]
            if items:
                lines.append(f"## {title}")
                lines += [f"- {x}" for x in items]
                lines.append("")
        return "\n".join(lines)

mem = Memory(embed)
mem.add_many(learned)
MEMORY_FILE.write_text(mem.to_markdown())

print("Asked 'which topics does the student struggle with?' → memory returned:")
for it in mem.recall("which LangChain topics does the student struggle with?", k=2):
    print(f"  [{it['type']}] {it['content']}")

Asked 'which topics does the student struggle with?' → memory returned:
  [semantic] Has been learning LangChain for 50 days
  [semantic] Struggles with LangGraph state


## Piece 3 — MCP tools: read the live LangChain docs (Days 35-36)

Instead of hand-written notes, the buddy studies from the **real, current LangChain docs** — served over MCP at the public `docs.langchain.com/mcp` endpoint (no API key). We load its tools with `langchain-mcp-adapters` and keep the read-only lookup ones.

In [4]:
mcp_client = MultiServerMCPClient({
    "langchain_docs": {"transport": "streamable_http", "url": "https://docs.langchain.com/mcp"}
})

all_tools = await mcp_client.get_tools()
docs_tools = [t for t in all_tools if "search" in t.name or "query" in t.name]  # skip submit_feedback

print("Connected to the LangChain docs MCP. Tools the buddy can call:")
for t in docs_tools:
    print(" •", t.name)

Connected to the LangChain docs MCP. Tools the buddy can call:
 • search_docs_by_lang_chain
 • query_docs_filesystem_docs_by_lang_chain


## Piece 4 — Skills: load a tutor skill

A **skill** is a markdown file that specializes how the agent behaves (the `load_skill` pattern from the series). We give the buddy a tutor skill: ground everything in the docs, one question at a time, short and example-first.

In [5]:
SKILL_DIR = Path("skills"); SKILL_DIR.mkdir(exist_ok=True)
(SKILL_DIR / "study_tutor.md").write_text(
"""# Skill: LangChain Study Tutor
You are a sharp, encouraging LangChain tutor.
- Ground everything you say in the LangChain docs you look up - never guess.
- Ask ONE focused question at a time.
- Keep it short and example-first.
- When the student is wrong, explain the key idea in two sentences, then let them retry.
""")

def load_skill(name):
    return (SKILL_DIR / f"{name}.md").read_text()

tutor_skill = load_skill("study_tutor")
print(tutor_skill)

# Skill: LangChain Study Tutor
You are a sharp, encouraging LangChain tutor.
- Ground everything you say in the LangChain docs you look up - never guess.
- Ask ONE focused question at a time.
- Keep it short and example-first.
- When the student is wrong, explain the key idea in two sentences, then let them retry.



## Piece 5 — The agent: a researcher that reads the docs (LangGraph + tool calling)

A LangGraph **ReAct agent** = the model + the MCP docs tools + the tutor skill. Give it a topic and it *decides* to call the docs tool, reads the result, and answers grounded in it. That's tool calling, live.

In [6]:
researcher = create_react_agent(model, docs_tools, prompt=tutor_skill)

demo = await researcher.ainvoke({"messages": [
    ("user", "In one line, what is LangGraph state? Look it up in the docs first.")]})
print("🤖", demo["messages"][-1].text.strip()[:400])

🤖 In LangGraph, **state** is the shared data structure (stored in named channels) that nodes read from and return updates to throughout graph execution.

Here is a quick Python example:

```python
from typing import TypedDict

class State(TypedDict):
    messages: list
    user_id: str
```

**Quick check:** If a node returns `{"user_id": "alice"}`, what happens to the rest of the fields in the state


## Piece 6 — Assemble the Study Buddy (+ judge + prompt caching)

One agent now wraps it all. When it drills a topic it **researches the docs once** (via the MCP agent) and caches that text — then reuses it as a stable prefix across the quiz, the explanation, and the grade. That reuse is the setup for **prompt caching** (Days 39-40): Gemini's implicit cache reuses a large repeated prefix (it shows up as `cache_read` in the usage). Our doc here is small, so you'll often see `0` — the *pattern* is the point; on big contexts it saves real tokens. The **judge** (Day 45) grades against the same doc.

In [7]:
class Grade(BaseModel):
    verdict: Literal["PASS", "FAIL"] = Field(description="PASS only if essentially correct")
    feedback: str = Field(description="one short, encouraging sentence")

class StudyBuddy:
    """LangGraph agent (MCP docs) + memory (LangMem) + RAG + skill + prompt caching + judge."""
    def __init__(self, model, memory, manager, researcher, skill):
        self.model = model
        self.judge = model.with_structured_output(Grade)
        self.mem = memory
        self.manager = manager
        self.researcher = researcher
        self.skill = skill
        self.notes = {}          # topic -> doc text, fetched once then REUSED (prompt caching)
        self.last_usage = None

    async def research(self, topic):
        """LangGraph agent pulls grounded notes from the real LangChain docs over MCP."""
        if topic not in self.notes:
            r = await self.researcher.ainvoke({"messages": [
                ("user", f"Look up '{topic}' in the LangChain docs and summarize the key idea a "
                         f"learner must know, in 4-5 sentences with one tiny example.")]})
            self.notes[topic] = r["messages"][-1].text.strip()
        return self.notes[topic]

    def _ask(self, prompt):
        resp = self.model.invoke(prompt)
        self.last_usage = resp.usage_metadata   # peek at prompt-cache token reuse
        return resp.text.strip()

    def greet(self):
        prefs = ", ".join(self.mem.preferences()) or "clear explanations"
        return self._ask("You are a warm LangChain study buddy. Your long-term memory about the student:\n\n"
                         f"{self.mem.to_markdown()}\n\nGreet them by name in 2 sentences, honor their preference "
                         f"for {prefs}, and offer to drill a LangChain topic they're weak at.")

    def focus(self):   # recall by meaning, but prefer an actual weakness
        for it in self.mem.recall("a LangChain topic the student struggles with and should practice", k=5):
            if "struggl" in it["content"].lower():
                return it["content"]
        return self.mem.recall("what should the student practice", k=1)[0]["content"]

    def quiz(self, topic):   # the fetched doc is the stable prefix reused across quiz/explain/grade
        return self._ask(f"{self.skill}\n\nUsing ONLY these LangChain docs:\n\n{self.notes[topic]}\n\n"
                         f"Ask ONE short conceptual question about what it fundamentally is and how it works. "
                         f"Question only, no answer.")

    def explain(self, topic):
        return self._ask(f"Using ONLY these LangChain docs:\n\n{self.notes[topic]}\n\n"
                         f"In 2 short sentences, explain the key idea to a student who just got it wrong.")

    def grade(self, topic, question, answer):
        return self.judge.invoke(f"LangChain docs:\n{self.notes[topic]}\n\nQuestion: {question}\n"
                                 f"Student answer: {answer}\n\nPASS if the answer is essentially correct about "
                                 f"the concept per the docs, even if worded differently than the question.")

    def learn(self, topic, recap):   # write new memory, retire the resolved weakness (Day 49)
        new = self.manager.invoke({"messages": [{"role": "user", "content": recap}]})
        self.mem.add_many(new)
        key = topic.split()[0].lower()
        self.mem.retire(lambda it: it["type"] == "semantic"
                        and "struggl" in it["content"].lower() and key in it["content"].lower())
        return new

buddy = StudyBuddy(model, mem, manager, researcher, tutor_skill)
print("🤖 Study buddy is ready")

🤖 Study buddy is ready


## A Real Study Session

### It greets you — and already knows your weak LangChain topics

In [8]:
print(buddy.greet())
print("\nMemory says focus on:", buddy.focus())

Hey Yash, huge congrats on shipping your RAG bot yesterday—that 50-day streak is really paying off! 

Ready to keep the momentum going with a quick, example-first drill on either LangGraph state or tool calling today?

Memory says focus on: Struggles with LangGraph state


### It researches the live docs (MCP) and quizzes you from them

In [9]:
topic = "LangGraph state"
notes = await buddy.research(topic)                 # LangGraph agent → LangChain docs MCP
print("📚 grounded straight from the docs:\n", notes[:260], "...\n")

question = buddy.quiz(topic)
print("❓", question)
print("   prompt-cache / token usage:", buddy.last_usage)

📚 grounded straight from the docs:
 Here is a quick look at how state is defined in LangGraph:

```python
import operator
from typing import Annotated, TypedDict

class State(TypedDict):
    count: int  # Overwrites value (last-write-wins)
    logs: Annotated[list[str], operator.add]  # Appends  ...

❓ If Node A returns `{"count": 5}` and Node B then returns `{"count": 10}`, what will `state["count"]` equal after both nodes complete?
   prompt-cache / token usage: {'input_tokens': 366, 'output_tokens': 1651, 'total_tokens': 2017, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 1613}}


### You give it a shaky answer — and it grades you honestly (Day 45 judge)

In [10]:
first = "Uh, state is just global variables I set anywhere in the graph?"
g = buddy.grade(topic, question, first)
print("📝", g.verdict, "—", g.feedback)

📝 FAIL — Not quite, but keep going—remember that `count` uses a last-write-wins rule, so Node B's return value of 10 will overwrite 5.


### It explains from the docs, you retry — and nail it

In [11]:
print("📖", buddy.explain(topic), "\n")

second = ("State is a shared, typed data structure - usually a TypedDict - passed between nodes. "
          "Each node reads it and returns updates that get merged back in via reducers, so the graph "
          "carries context forward step to step.")
g2 = buddy.grade(topic, question, second)
print("📝", g2.verdict, "—", g2.feedback)
print("   prompt-cache / token usage:", buddy.last_usage)

📖 After both nodes complete, `state["count"]` will equal **10**. 

Because `count` does not have an attached reducer, it uses the default last-write-wins strategy to overwrite existing values with the newest update. 

📝 FAIL — Make sure to directly answer the specific question about `state["count"]` equaling 10 due to the last-write-wins strategy!
   prompt-cache / token usage: {'input_tokens': 286, 'output_tokens': 529, 'total_tokens': 815, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 480}}


### It updates its own memory and retires the topic

You passed — so it writes a fresh memory (LangMem) and **drops LangGraph state from your weak list**.

In [12]:
print("BEFORE:\n" + mem.to_markdown())

recap = ("Yash just correctly explained LangGraph state - a shared typed object passed between nodes, "
         "updated via reducers. He is confident with it now.")
new = buddy.learn(topic, recap) if g2.verdict == "PASS" else []
MEMORY_FILE.write_text(mem.to_markdown())

print("the buddy wrote:", [m.content.content for m in new])
print("\nAFTER:\n" + mem.to_markdown())

BEFORE:
# 🧠 What the buddy remembers about you

## Facts about you (semantic)
- Name is Yash
- Has been learning LangChain for 50 days
- Struggles with LangGraph state
- Struggles with tool calling

## Things that happened (episodic)
- Shipped their first RAG bot yesterday

## How to help you (procedural)
- Prefers short, example-first explanations

the buddy wrote: []

AFTER:
# 🧠 What the buddy remembers about you

## Facts about you (semantic)
- Name is Yash
- Has been learning LangChain for 50 days
- Struggles with LangGraph state
- Struggles with tool calling

## Things that happened (episodic)
- Shipped their first RAG bot yesterday

## How to help you (procedural)
- Prefers short, example-first explanations



### A new session — it moves you on to your next weak topic

Close the laptop, come back tomorrow. Only `day50_memory.md` survives. A fresh model reads it and now steers toward **tool calling**, because LangGraph state is handled. The buddy compounds.

In [16]:
fresh = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
saved = MEMORY_FILE.read_text()

reply = fresh.invoke([
    ("system", f"You are a LangChain study buddy. Your long-term memory about the student:\n\n{saved}\n\nUse it to personalize."),
    ("human", "I'm back. What should we study today? Keep it short."),
])
print("🤖", reply.text.strip())

🤖 Welcome back, Yash! Congrats again on shipping that RAG bot yesterday. 🎉

Since we're building momentum, I suggest we tackle **Tool Calling** or **LangGraph State** today to iron out those tricky spots.

For instance, we can start with **LangGraph State** by building a minimal state schema:

```python
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage
import operator

class State(TypedDict):
    # Appends new messages to history instead of overwriting
    messages: Annotated[list[BaseMessage], operator.add]
```

Which one do you want to conquer today: **LangGraph State** or **Tool Calling**?


## Bonus — Give It a Voice (Day 48)

The same buddy, out loud, through Gemini TTS.

In [14]:
from google import genai
from google.genai import types
from IPython.display import Audio

voice_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def say(text, voice="Kore"):
    r = voice_client.models.generate_content(
        model="gemini-2.5-flash-preview-tts", contents=text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=voice)))))
    pcm = r.candidates[0].content.parts[0].inline_data.data
    return Audio(np.frombuffer(pcm, dtype=np.int16), rate=24000)

try:
    display(say("Nice work, Yash - LangGraph state is solid now. Next time, let's take on tool calling."))
except Exception as e:
    print("(Voice is a preview model with its own quota - skipped:", type(e).__name__, ")")

## Key Takeaways

1. **An agent is the pieces, assembled.** Memory (Day 49) + RAG (Days 8-14) + MCP docs + tool calling (Days 35-36) + skills + prompt caching (Days 39-40) + judge (Day 45) + voice (Day 48) — none new today, but together they're a buddy that teaches you LangChain itself.
2. **Ground it in real, live sources.** Over MCP the buddy reads the *current* LangChain docs — so it never quizzes you from a stale guess.
3. **It compounds.** Every session writes new memory and retires what you've mastered, so tomorrow's buddy is smarter than today's.

---

## That's a wrap 🎉

**50 days. 50 concepts.** From the first API call to a LangGraph agent that reads the docs, remembers you, grades you, talks — and keeps teaching you LangChain long after Day 50.

Thank you for following along. Now go build.